# 03 V-Dem

**Owner:** Nicolas

**What this notebook does:** keeps the democracy index and regime type per country-year, adds readable regime labels and builds the regime periods for the timeline.

**Reads:** `vdem/V-Dem-CY-Core-v16.csv`

**Writes:** `vdem.csv`, `vdem_regime_spans.csv`

This is a main notebook. It must run from top to bottom with Kernel → Restart and Run All.
Write simple code in small steps: one operation per line, print the result after each step. Your sandbox section for this notebook: **sandbox_nicolas, Part A**. Paste only the working lines, one step per cell, with a short note above each.

## Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW = "../data/raw/"
CLEAN = "../data/clean/"
pd.set_option("display.max_columns", 50)

## A1 Load only the columns you need

Paste your working code from the sandbox. Add one sentence above it that says what it does.

In [ ]:
# your code here


vdem_full = pd.read_csv(RAW + "vdem/V-Dem-CY-Core-v16.csv")
print(vdem_full.shape)

cols = ["country_name", "country_text_id", "COWcode", "year", "v2x_polyarchy", "v2x_regime"]
vdem_raw = vdem_full[cols].copy()
print(vdem_raw.shape)
print(vdem_raw.head())

## A2 Filter years and rename

Paste your working code from the sandbox. Add one sentence above it that says what it does.

In [ ]:
# your code here


from_1950 = vdem_raw["year"] >= 1950
until_2025 = vdem_raw["year"] <= 2025

vdem = vdem_raw[from_1950 & until_2025].copy()
print(vdem_raw.shape)
print(vdem.shape)

new_names = {"country_text_id": "iso3", "COWcode": "cow_code", "v2x_polyarchy": "democracy_index", "v2x_regime": "regime_type"}
vdem = vdem.rename(columns=new_names)
print(vdem.columns)

print(vdem[vdem["regime_type"].isna()])
vdem = vdem.dropna(subset=["regime_type"]).copy()
print(vdem.shape)

vdem["regime_type"] = vdem["regime_type"].astype(int)
print(vdem["regime_type"].unique())

## A3 Regime labels

Paste your working code from the sandbox. Add one sentence above it that says what it does.

In [ ]:
# your code here

labels = {0: "closed autocracy", 1: "electoral autocracy", 2: "electoral democracy", 3: "liberal democracy"}

regime_label = []
for code in vdem["regime_type"]:
    regime_label.append(labels[code])

vdem["regime_label"] = regime_label
print(vdem[["regime_type", "regime_label"]].head())
print(vdem["regime_label"].value_counts())

## A4 Regime periods

Paste your working code from the sandbox. Add one sentence above it that says what it does.

In [ ]:
# your code here


vdem = vdem.sort_values(["iso3", "year"]).copy()

periods = []

for code, rows in vdem.groupby("iso3"):
    start = None
    current = None
    prev_year = None

    for year, regime_type in zip(rows["year"], rows["regime_label"]):
        if prev_year is None:         # the very first row: a period starts
            start = year
            current = regime_type
        elif regime_type != current or year != prev_year + 1:     # regime changed or a year is missing: the old period ends
            periods.append({"iso3": code, "regime_label": current, "start": start, "end": prev_year})
            start = year
            current = regime_type
        prev_year = year

    periods.append({"iso3": code, "regime_label": current, "start": start, "end": prev_year})

vdem_regime_spans = pd.DataFrame(periods)
print(vdem_regime_spans[vdem_regime_spans["iso3"] == "CHE"])

## A5 Save

Paste your working code from the sandbox. Add one sentence above it that says what it does.

In [ ]:
# your code here


vdem.to_csv(CLEAN + "vdem.csv", index=False)
vdem_regime_spans.to_csv(CLEAN + "vdem_regime_spans.csv", index=False)

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
vdem_check = pd.read_csv(CLEAN + "vdem.csv")
spans_check = pd.read_csv(CLEAN + "vdem_regime_spans.csv")
assert vdem_check.duplicated(["iso3", "year"]).sum() == 0
assert vdem_check["year"].between(1950, 2025).all()
for col in ["iso3", "country_name", "cow_code", "year", "democracy_index", "regime_type", "regime_label"]:
    assert col in vdem_check.columns, f"missing {col}"
assert vdem_check["regime_type"].dtype.kind == "i"
assert list(spans_check.columns) == ["iso3", "regime_label", "start", "end"]
assert (spans_check["end"] - spans_check["start"] + 1).sum() == len(vdem_check)
print("OK")

## Decisions in this notebook

Write every choice you made here (and copy it into the decisions log in `CONVENTIONS.md`). You need these for the report.

| Decision | Why |
| --- | --- |
|  |  |